In [1]:
import pandas as pd
import numpy as np

In [3]:
gva_shootings_2023 = pd.read_csv("gunviolencearchive_2023.csv") # unfortunately the operations column
# ended up as NaN for all - in the web archive "Operations" has links to incident details (drive-by, etc) 
# as well as the ages of the people involved, weapon type, where it took place (bar/club, etc)
print(gva_shootings_2023.shape)
gva_shootings_2023.head()

(2000, 11)


,Incident ID,Incident Date,State,City Or County,Address,Victims Killed,Victims Injured,Suspects Killed,Suspects Injured,Suspects Arrested,Operations
0,2792171,"December 31, 2023",Ohio,Cincinnati,7800 block of Dawn Rd,1,0,0,0,1,NaN
1,2792333,"December 31, 2023",New Hampshire,Manchester,274 Amherst St,1,0,0,0,1,NaN
2,2790984,"December 31, 2023",Tennessee,Memphis,3302 block of River Valley Ln,1,0,0,0,0,NaN
3,2793054,"December 31, 2023",Indiana,Fredericksburg,6000 block of W Mt Carmel Rd,1,0,0,0,0,NaN
4,2790802,"December 31, 2023",New Mexico,Albuquerque,6600 block of Moore Dr SW,1,0,0,0,1,NaN


In [11]:
gva_shootings_2023[gva_shootings_2023["Victims Killed"] + gva_shootings_2023["Victims Injured"] > 4].head()
# gun violence archive defines a mass shooting as any where victims killed/injured exceeds 4 and takes
# place in one location

,Incident ID,Incident Date,State,City Or County,Address,Victims Killed,Victims Injured,Suspects Killed,Suspects Injured,Suspects Arrested,Operations
8,2790854,"December 31, 2023",California,Hawthorne,14125 Crenshaw Blvd,1,5,0,0,0,NaN
75,2790633,"December 30, 2023",Texas,Beaumont,6500 block of Bigner Rd,1,4,0,0,0,NaN
337,2786002,"December 24, 2023",Florida,Orlando (Lockhart),2800 block of Sudman Way,2,3,0,0,0,NaN
384,2786038,"December 24, 2023",Texas,Houston,5828 Martin Luther King Blvd,1,4,0,0,0,NaN
387,2787693,"December 24, 2023",Texas,Houston,8627 Glenvista St,1,4,0,0,1,NaN


In [108]:
mj_shootings = pd.read_csv("Mother Jones - Mass Shootings Database, 1982 - 2024 - Sheet1.csv")
mj_shootings = mj_shootings.drop(columns=["sources", "mental_health_sources", "latitude", "longitude", "sources_additional_age", "mental_health_details", "summary"])
print(mj_shootings.shape)
mj_shootings.head()

(151, 17)


,case,location,date,fatalities,injured,total_victims,location.1,age_of_shooter,prior_signs_mental_health_issues,weapons_obtained_legally,where_obtained,weapon_type,weapon_details,race,gender,type,year
0,Apalachee High School shooting,"Winder, Georgia",9/4/24,4,9,13,School,14,yes,-,-,semiautomatic rifle,AR-15,White,M,mass,2024
1,Arkansas grocery store shooting,"Fordyce, Arkansas",6/21/24,4,10,14,workplace,44,-,-,-,shotgun; semiautomatic pistol,12-gauge shotgun,White,M,mass,2024
2,UNLV shooting,"Las Vegas, Nevada",12/6/23,3,1,4,School,67,-,-,-,semiautomatic handgun,-,White,M,mass,2023
3,Maine bowling alley and bar shootings,"Lewiston, Maine",10/25/23,18,13,31,Other,40,yes,-,Yes,semiautomatic rifle,AR-15-style rifle (Rugar SFAR),White,M,Spree,2023
4,Jacksonville Dollar General store shooting,"Jacksonville, Florida",8/26/23,3,0,3,workplace,21,yes,yes,local gun stores,"semiautomatic rifle, semiautomatic handgun",AR-15-style rifle; Glock pistol,White,M,mass,2023


In [80]:
mj_shootings.loc[61]["where_obtained"]

'Unclear; the firearm was stolen in Utah. A second handgun Lam had (also stolen) was unused in the attack.'

In [109]:
def how_obtained(s, method_strs): # method_strs is an array
    for method in method_strs:
        if method in s.lower():
            return True
    return False

stolen = mj_shootings.loc[61]["where_obtained"]
how_obtained(stolen, "stolen")

True

In [110]:
stolen = mj_shootings["where_obtained"].apply(lambda x: how_obtained(x, ["taken", "stolen"])).to_numpy()
mj_shootings["weapon_stolen"] = stolen

In [111]:
illegally_bought = ["purchased", "shop", "pawn", "internet"]
purchased = mj_shootings["where_obtained"].apply(lambda x: how_obtained(x, illegally_bought)).to_numpy()
mj_shootings["weapon_bought_illegally"] = purchased

In [114]:
mj_shootings["weapons_obtained_legally"] = mj_shootings["weapons_obtained_legally"].replace({"Yes": True, "No": False})

In [115]:
mj_shootings.tail(10)

,case,location,date,fatalities,injured,total_victims,location.1,age_of_shooter,prior_signs_mental_health_issues,weapons_obtained_legally,where_obtained,weapon_type,weapon_details,race,gender,type,year,weapon_stolen,weapon_bought_illegally
141,Luby's massacre,"Killeen, Texas",10/16/1991,24,20,44,Other,35,No,True,"Mike's Gun Shop in Henderson, Nev.",Two semiautomatic handguns,"9mm Glock 17, 9mm Ruger P89 semiautomatic hand...",white,Male,Mass,1991,False,True
142,GMAC massacre,"Jacksonville, Florida",6/18/1990,10,4,14,Other,42,No,True,Unknown,"One rifle, one revolver",.30-caliber Universal M1 carbine rifle; .38-ca...,black,Male,Mass,1990,False,False
143,Standard Gravure shooting,"Louisville, Kentucky",9/14/1989,9,12,21,Workplace,47,Yes,True,AK-47 purchased from Tilford's Gun Sales in Lo...,"Three semiautomatic handguns (two assault), on...","Two Intratec MAC-11, 9mm SIG Sauer semiautomat...",white,Male,Mass,1989,False,True
144,Stockton schoolyard shooting,"Stockton, California",1/17/1989,6,29,35,School,26,Yes,True,"Sandy Trading Post in Sandy, Ore.; Hunter Loan...","One semiautomatic handgun, one rifle (assault)",9mm Taurus semiautomatic handgun; AK-47 Chines...,white,Male,Mass,1989,False,False
145,ESL shooting,"Sunnyvale, California",2/16/1988,7,4,11,Workplace,39,Yes,True,Various sporting goods and gun stores in North...,"Two semiautomatic handguns, one rifle, two rev...",".380 ACP Browning, 9mm Smith & Wesson semiauto...",white,Male,Mass,1988,False,False
146,Shopping centers spree killings,"Palm Bay, Florida",4/23/1987,6,14,20,Other,59,Yes,True,"Gun store in Norwood, Ohio; The Oaks Trading P...","One rifle, one revolver, one shotgun","Sturm, Ruger Mini-14 semiautomatic rifle; 20-g...",white,Male,Spree,1987,False,False
147,United States Postal Service shooting,"Edmond, Oklahoma",8/20/1986,15,6,21,Workplace,44,Unclear,True,"Issued by Oklahoma National Guard, where Sherr...",Three semiautomatic handguns,".22-caliber, two .45-caliber Colt Model 1911-A...",white,Male,Mass,1986,False,False
148,San Ysidro McDonald's massacre,"San Ysidro, California",7/18/1984,22,19,41,Other,41,Yes,True,Unknown,"One semiautomatic handgun, one rifle (assault)...",9mm Browning P35 Hi-Power semiautomatic handgu...,white,Male,Mass,1984,False,False
149,Dallas nightclub shooting,"Dallas, Texas",6/29/1984,6,1,7,Other,39,Yes,False,"Hines Boulevard Pawn Shop in Dallas, Texas",One semiautomatic handgun,9mm Smith & Wesson 459 semiautomatic handgun,white,Male,Mass,1984,False,True
150,Welding shop shooting,"Miami, Florida",8/20/1982,8,3,11,Other,51,Yes,True,"Garcia Gun Center in Hialeah, Fla.",One shotgun,Mossberg 500 Persuader pump-action shotgun wit...,white,Male,Mass,1982,False,False
